In [1]:
import pandas as pd
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, AutoModelForSequenceClassification, AutoTokenizer, set_seed
from datasets import Dataset
import joblib
import numpy as np
from google.colab import drive
import os
import json
import zipfile
from sklearn.metrics import f1_score, classification_report
import time
import torch
from transformers.trainer_utils import get_last_checkpoint

In [2]:
drive.mount('/content/drive')

SPLIT_PATH = "/content/drive/MyDrive/thesis_results/SCOTBESS_splits/SCOTBESS_FULL_ANNOTATED_TERRA_LOW_SPLIT.csv"
MLB_PATH = "/content/drive/MyDrive/thesis_results/SCOTBESS_splits/scotbess_mlb.joblib"

output_dir = "/content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_DistilBERT_Bitfit"
os.makedirs(output_dir, exist_ok=True)


Mounted at /content/drive


Loading the dataset

In [3]:
scotbess_df = pd.read_csv(SPLIT_PATH)

In [4]:
scotbess_df_train = scotbess_df[scotbess_df["split"] == "train"].reset_index(drop=True)
scotbess_df_val = scotbess_df[scotbess_df["split"] == "validation"].reset_index(drop=True)
scotbess_df_test = scotbess_df[scotbess_df["split"] == "test"].reset_index(drop=True)

print("Train shape:", scotbess_df_train.shape)
print("Validation shape:", scotbess_df_val.shape)
print("Test shape:", scotbess_df_test.shape)

Train shape: (1340, 7)
Validation shape: (165, 7)
Test shape: (170, 7)


In [5]:
mlb = joblib.load(MLB_PATH)

print("Number of labels:", len(mlb.classes_))
print(mlb.classes_)

Number of labels: 20
['Agricultural Land' 'Community and Economic Benefits'
 'Consultation, Transparency and Information' 'Cumulative Impact'
 'Decommissioning and Site Restoration' 'Emergency Planning and Response'
 'Fire and Explosion Risk' 'Grid Connection and Electrical Infrastructure'
 'Health and Wellbeing' 'Landscape, Visual and Heritage Impact'
 'Light Pollution' 'Noise' 'Planning Policy and Regulatory Compliance'
 'Project Need' 'Property Value'
 'Residential Proximity and Separation Distance' 'Site Selection'
 'Traffic' 'Water and Soil Contamination' 'Wildlife and Ecology']


In [6]:
def parse_labels(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    return json.loads(value)

for df in [scotbess_df_train, scotbess_df_val, scotbess_df_test]:
    df["label_list"] = df["labels"].apply(parse_labels)


In [7]:
scotbess_y_train = mlb.transform(scotbess_df_train["label_list"])
scotbess_y_val = mlb.transform(scotbess_df_val["label_list"])
scotbess_y_test = mlb.transform(scotbess_df_test["label_list"])


In [8]:
scotbess_y_train.shape, scotbess_y_val.shape, scotbess_y_test.shape


((1340, 20), (165, 20), (170, 20))

In [9]:
scotbess_X_train = scotbess_df_train["final_masked_text"].fillna("").astype(str)
scotbess_X_val = scotbess_df_val["final_masked_text"].fillna("").astype(str)
scotbess_X_test = scotbess_df_test["final_masked_text"].fillna("").astype(str)


In [10]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [11]:
# for DistilBER max token length is 512 -  truncation will happen
def tokenize(texts):
    return tokenizer(texts.tolist(), padding="max_length", truncation=True, max_length=512)

train_enc = tokenize(scotbess_X_train)
dev_enc = tokenize(scotbess_X_val)
test_enc = tokenize(scotbess_X_test)


In [13]:
y_train_bin = scotbess_y_train.astype(np.float32)
y_dev_bin = scotbess_y_val.astype(np.float32)
y_test_bin = scotbess_y_test.astype(np.float32)

print(mlb.classes_)
print(y_train_bin.shape)


['Agricultural Land' 'Community and Economic Benefits'
 'Consultation, Transparency and Information' 'Cumulative Impact'
 'Decommissioning and Site Restoration' 'Emergency Planning and Response'
 'Fire and Explosion Risk' 'Grid Connection and Electrical Infrastructure'
 'Health and Wellbeing' 'Landscape, Visual and Heritage Impact'
 'Light Pollution' 'Noise' 'Planning Policy and Regulatory Compliance'
 'Project Need' 'Property Value'
 'Residential Proximity and Separation Distance' 'Site Selection'
 'Traffic' 'Water and Soil Contamination' 'Wildlife and Ecology']
(1340, 20)


In [14]:
id2label = {i: label for i, label in enumerate(mlb.classes_)}
label2id = {label: i for i, label in enumerate(mlb.classes_)}

In [15]:
train_dataset = Dataset.from_dict({
    "input_ids": train_enc["input_ids"],
    "attention_mask": train_enc["attention_mask"],
    "labels": y_train_bin.astype("float32")})

eval_dataset = Dataset.from_dict({
    "input_ids": dev_enc["input_ids"],
    "attention_mask": dev_enc["attention_mask"],
    "labels": y_dev_bin.astype("float32")})

test_dataset = Dataset.from_dict({
    "input_ids": test_enc["input_ids"],
    "attention_mask": test_enc["attention_mask"],
    "labels": y_test_bin.astype("float32")})

In [16]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))   # sigmoid
    preds = (probs >= 0.5).astype(int)

    labels = labels.astype(int)

    f1_micro = f1_score(labels, preds, average="micro", zero_division=0)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)

    return {
        "f1_micro": f1_micro,
        "f1_macro": f1_macro}

In [17]:
print("Train labels:", y_train_bin.shape)
print("Val labels:", y_dev_bin.shape)
print("Test labels:", y_test_bin.shape)
print("Number of labels:", len(mlb.classes_))

Train labels: (1340, 20)
Val labels: (165, 20)
Test labels: (170, 20)
Number of labels: 20


In [18]:
def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def reset_cuda_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

#measure vram only for final best config
def get_peak_vram_gb():
    if not torch.cuda.is_available():
        return None

    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / (1024 ** 3)


def run_training(config, seed=0, evaluate_test=False, measure_vram=False, save_report=False):
    set_seed(seed)


    if measure_vram:
        reset_cuda_peak_memory()


    model = AutoModelForSequenceClassification.from_pretrained(
        config["base_model"],
        num_labels=len(mlb.classes_),
        problem_type="multi_label_classification",
        id2label=id2label,
        label2id=label2id)

  #BitFit: everything is frozen except bias terms and classifier head
    for name, param in model.named_parameters():
       param.requires_grad = False

    for name, param in model.named_parameters():
       if "bias" in name or "classifier" in name or "pre_classifier" in name:
           param.requires_grad = True



    #for the final statistics, for bitfit it will only reflect the bias terms + classification head
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    training_args = TrainingArguments(
        output_dir=config["output_dir"],

        learning_rate=config["learning_rate"],
        per_device_train_batch_size=config["batch_size"],
        per_device_eval_batch_size=config["batch_size"],

        num_train_epochs=config["num_train_epochs"],
        weight_decay=config["weight_decay"],
        warmup_ratio=config["warmup_ratio"],

        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,

        save_total_limit=2,
        fp16=True,
        report_to="none")

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=config["early_stopping_patience"])])


#for resuming if something goes wrong
    last_checkpoint = None
    if os.path.isdir(config["output_dir"]):
        last_checkpoint = get_last_checkpoint(config["output_dir"])

    if last_checkpoint is not None:
        print(f"Resuming from checkpoint: {last_checkpoint}")
    else:
        print("Starting training from scratch.")


    sync_cuda()
    train_start = time.perf_counter()
#includes training + epoch validation + checkpoint saving + early stopping + loading best model
    trainer.train(resume_from_checkpoint=last_checkpoint)

    sync_cuda()
    train_time_sec = time.perf_counter() - train_start
    if measure_vram:
        training_peak_vram_gb = get_peak_vram_gb()
    else:
        training_peak_vram_gb = None

    sync_cuda()
    val_start = time.perf_counter()

    val_results = trainer.evaluate(eval_dataset, metric_key_prefix="val")

    sync_cuda()
    val_eval_time_sec = time.perf_counter() - val_start

    result = {
        "model": config["base_model"],
        "dataset": "Scot-BESS",
        "method": "bitfit",
        "seed": seed,

        "learning_rate": config["learning_rate"],
        "batch_size": config["batch_size"],
        "num_train_epochs": config["num_train_epochs"],

        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_metric": trainer.state.best_metric,
        "actual_epochs_trained": trainer.state.epoch,


        "train_time_sec": train_time_sec,
        "val_eval_time_sec": val_eval_time_sec,

        "training_peak_vram_gb": training_peak_vram_gb,

        "trainable_params": trainable_params,
        "total_params": total_params,

        "val_f1_macro": val_results["val_f1_macro"],
        "val_f1_micro": val_results["val_f1_micro"]}

    if evaluate_test:
        sync_cuda()
        test_start = time.perf_counter()

        # single forward pass — gives metrics + raw predictions
        test_pred_output = trainer.predict(test_dataset)

        sync_cuda()
        test_eval_time_sec = time.perf_counter() - test_start

        # derive predictions - needed for classification report
        test_probs = 1 / (1 + np.exp(-test_pred_output.predictions))
        test_binary_preds = (test_probs >= 0.5).astype(int)
        gold_labels = (test_pred_output.label_ids >= 0.5).astype(int)

        test_metrics = test_pred_output.metrics

        result.update({
            "test_eval_time_sec": test_eval_time_sec,
            "test_inference_per_sample_ms": (test_eval_time_sec / len(test_dataset)) * 1000,
            "test_f1_macro": test_metrics["test_f1_macro"],
            "test_f1_micro": test_metrics["test_f1_micro"],
            "avg_predicted_labels": float(test_binary_preds.sum(axis=1).mean()),
            "avg_gold_labels": float(gold_labels.sum(axis=1).mean()),})

        if save_report:
            report_dict = classification_report(
                gold_labels, test_binary_preds,
                target_names=mlb.classes_, zero_division=0, output_dict=True)
            report_df = pd.DataFrame(report_dict).T
            report_path = os.path.join(output_dir, f"classification_report_seed_{seed}.csv")
            report_df.to_csv(report_path)
            print(f"Classification report saved to {report_path}")

            #saving raw arrays for possible future analysis
            predictions_path = os.path.join(
                output_dir,
                f"test_predictions_seed_{seed}.npz")
            np.savez_compressed(
                predictions_path,
                y_true=gold_labels,
                y_pred=test_binary_preds,
                y_prob=test_probs,
                label_names=np.array(mlb.classes_),
                threshold=np.array([0.5]))
            result["test_predictions_path"] = predictions_path
            print(f"Predictions saved to {predictions_path}")

    result["total_measured_time_sec"] = (result["train_time_sec"] + result["val_eval_time_sec"] + result.get("test_eval_time_sec", 0))

    return result

In [19]:
#fixed params
base_config = {
    "output_dir": os.path.join(output_dir, "search"),
    "base_model": "distilbert-base-uncased",
    "tokenizer_name": "distilbert-base-uncased",

    "max_length": 512,
    "num_train_epochs": 10,
    "weight_decay": 0.00, #no weight decay for BitFit
    "warmup_ratio": 0.1,
    "early_stopping_patience": 3}

#small search on the most relevant hyperparameters
#higher learning rates than for FFT - the reduced number of trainable parameters requires larger updates to effectively adapt the model
learning_rates =  [1e-4, 5e-4, 1e-3] #FFT: 1e-5, 2e-5, 3e-5
batch_sizes = [8, 16]


search_results_path = os.path.join(output_dir, "search_results.csv")
search_results = []

for lr in learning_rates:
    for bs in batch_sizes:
        config = base_config.copy()
        config["learning_rate"] = lr
        config["batch_size"] = bs
        config["output_dir"] = (f"{base_config['output_dir']}/lr_{lr}_bs_{bs}")

        #skipingp already-completed configs on resume
        if os.path.exists(search_results_path):
            existing = pd.read_csv(search_results_path)
            already_done = existing[
                (existing["learning_rate"] == lr) &
                (existing["batch_size"] == bs)]
            if len(already_done) > 0:
                print(f"Skipping lr={lr}, bs={bs} (already done)")
                search_results.append(already_done.iloc[0].to_dict())
                continue

        print("=" * 80)
        print(f"Running DistilBERT: lr={lr}, batch_size={bs}")
        print("=" * 80)

        result = run_training(config, seed=0)
        search_results.append(result)

        #saving incrementally after every config
        pd.DataFrame(search_results).to_csv(search_results_path, index=False)

search_results_df = pd.DataFrame(search_results)
search_results_df = search_results_df.sort_values("val_f1_macro", ascending=False).reset_index(drop=True)
search_results_df.to_csv(search_results_path, index=False)
#!!! train_time_sec in search results is unreliable due to checkpoint resumption !!!
# !!!timing is only reported from the final seed runs - I ensured the run is not resumed
search_results_df

Running DistilBERT: lr=0.0001, batch_size=8


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro
0.208336,0.299395,10,0.783107,0.713298


,model,dataset,method,seed,learning_rate,batch_size,num_train_epochs,best_checkpoint,best_metric,actual_epochs_trained,train_time_sec,val_eval_time_sec,training_peak_vram_gb,trainable_params,total_params,val_f1_macro,val_f1_micro,total_measured_time_sec
0,distilbert-base-uncased,Scot-BESS,bitfit,0,0.0010,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.722972,10.0,178.112976,0.835315,None,657428,66968852,0.722972,0.787190,178.948291
1,distilbert-base-uncased,Scot-BESS,bitfit,0,0.0010,16,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.713298,10.0,172.847519,0.740705,None,657428,66968852,0.713298,0.783107,173.588223
2,distilbert-base-uncased,Scot-BESS,bitfit,0,0.0005,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.705177,10.0,178.983541,0.889624,None,657428,66968852,0.705177,0.780231,179.873165
3,distilbert-base-uncased,Scot-BESS,bitfit,0,0.0005,16,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.693635,10.0,172.128691,0.799602,None,657428,66968852,0.693635,0.777246,172.928292
4,distilbert-base-uncased,Scot-BESS,bitfit,0,0.0001,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.543637,10.0,175.416986,0.895292,None,657428,66968852,0.543637,0.681257,176.312278
5,distilbert-base-uncased,Scot-BESS,bitfit,0,0.0001,16,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.503120,10.0,170.081846,0.797966,None,657428,66968852,0.503120,0.650658,170.879812


In [20]:
best_row = search_results_df.iloc[0]

best_lr = float(best_row["learning_rate"])
best_batch_size = int(best_row["batch_size"])

print("Best learning rate:", best_lr)
print("Best batch size:", best_batch_size)
print("Best validation macro-F1:", best_row["val_f1_macro"])
print("Best checkpoint:", best_row["best_checkpoint"])

Best learning rate: 0.001
Best batch size: 8
Best validation macro-F1: 0.7229719706566305
Best checkpoint: /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_DistilBERT_Bitfit/search/lr_0.001_bs_8/checkpoint-1512


In [21]:
best_config = base_config.copy()
best_config["learning_rate"] = best_lr
best_config["batch_size"] = best_batch_size
best_config["selection_metric"] = "val_f1_macro"
best_config["best_validation_macro_f1"] = float(best_row["val_f1_macro"])
best_config["best_validation_micro_f1"] = float(best_row["val_f1_micro"])
best_config["best_checkpoint_from_search"] = best_row["best_checkpoint"]

best_config_path = os.path.join(output_dir, "best_config.json")

with open(best_config_path, "w") as f:
    json.dump(best_config, f, indent=2)


**Final run (test set) on the best found configuration**

In [22]:
with open(best_config_path, "r") as f:
    final_config = json.load(f)

In [23]:
test_output_dir = os.path.join(output_dir, "test")
os.makedirs(test_output_dir, exist_ok=True)

test_results_path = os.path.join(test_output_dir, "SCOTBESS_DistilBERT_BitFit_test_results.csv")
test_results = []

for seed in [0, 1, 2]:
    config = final_config.copy()
    config["seed"] = seed
    config["output_dir"] = (os.path.join(test_output_dir, f"SCOTBESS_DistilBERT_BitFit_test_seed_{seed}"))

    # Skip already-completed seeds on resume
    if os.path.exists(test_results_path):
        existing = pd.read_csv(test_results_path)
        already_done = existing[existing["seed"] == seed]
        if len(already_done) > 0:
            print(f"Skipping seed={seed} (already done)")
            test_results.append(already_done.iloc[0].to_dict())
            continue

    print("=" * 80)
    print(f"Final run: seed={seed}, lr={final_config['learning_rate']}, batch_size={final_config['batch_size']}")
    print("=" * 80)
    result = run_training(config, seed=seed, evaluate_test=True, measure_vram=True, save_report = True)
    test_results.append(result)

    # Save incrementally after every seed
    pd.DataFrame(test_results).to_csv(test_results_path, index=False)

test_results_df = pd.DataFrame(test_results)
test_results_df.to_csv(test_results_path, index=False)
test_results_df

Final run: seed=0, lr=0.001, batch_size=8


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting training from scratch.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.529110,0.426848,0.609613,0.420991
2,0.388964,0.352466,0.719871,0.581833
3,0.332104,0.336544,0.724119,0.589672
4,0.293359,0.328137,0.755602,0.679730
5,0.264166,0.313307,0.764458,0.675748
6,0.236556,0.319058,0.764420,0.685636
7,0.213762,0.298441,0.778400,0.694449
8,0.193790,0.307138,0.784314,0.703711
9,0.176757,0.299219,0.786157,0.719655
10,0.163034,0.300592,0.788100,0.723047


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro
0.161068,0.299801,10,0.788732,0.724583


Classification report saved to /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_DistilBERT_Bitfit/classification_report_seed_2.csv
Predictions saved to /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_DistilBERT_Bitfit/test_predictions_seed_2.npz


,model,dataset,method,seed,learning_rate,batch_size,num_train_epochs,best_checkpoint,best_metric,actual_epochs_trained,...,val_f1_macro,val_f1_micro,test_eval_time_sec,test_inference_per_sample_ms,test_f1_macro,test_f1_micro,avg_predicted_labels,avg_gold_labels,test_predictions_path,total_measured_time_sec
0,distilbert-base-uncased,Scot-BESS,bitfit,0,0.001,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.723047,10.0,...,0.723047,0.788100,0.801211,4.713008,0.715289,0.781282,5.647059,5.917647,/content/drive/MyDrive/thesis_results/SCOTBESS...,178.987378
1,distilbert-base-uncased,Scot-BESS,bitfit,1,0.001,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.723191,10.0,...,0.723191,0.787377,0.866236,5.095506,0.708150,0.778171,5.723529,5.917647,/content/drive/MyDrive/thesis_results/SCOTBESS...,179.039819
2,distilbert-base-uncased,Scot-BESS,bitfit,2,0.001,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.724583,10.0,...,0.724583,0.788732,0.778730,4.580765,0.702796,0.776363,5.629412,5.917647,/content/drive/MyDrive/thesis_results/SCOTBESS...,177.897182


In [24]:
test_summary_df = test_results_df[[
    "test_f1_macro",
    "test_f1_micro",
    "avg_predicted_labels",
    "avg_gold_labels",
    "train_time_sec",
    "val_eval_time_sec",
    "test_eval_time_sec",
    "training_peak_vram_gb",
    "trainable_params",
    "total_params",
    "actual_epochs_trained",

    "total_measured_time_sec"]].agg(["mean", "std"])

test_summary_path =  os.path.join(test_output_dir, "SCOTBESS_DistilBERT_BitFit_test_results_summary.csv")
test_summary_df.to_csv(test_summary_path)

test_summary_df

,test_f1_macro,test_f1_micro,avg_predicted_labels,avg_gold_labels,train_time_sec,val_eval_time_sec,test_eval_time_sec,training_peak_vram_gb,trainable_params,total_params,actual_epochs_trained,total_measured_time_sec
mean,0.708745,0.778605,5.666667,5.917647,176.980417,0.845650,0.815392,0.903787,657428.0,66968852.0,10.0,178.641459
std,0.006267,0.002488,0.050029,0.000000,0.570191,0.055987,0.045444,0.000672,0.0,0.0,0.0,0.645096
